# Mini-Project 1: UBOS District Population Forecaster

**Purpose:** Compare three forecasting methods for five Ugandan districts, evaluate them on a held-out period, forecast 2025–2029, and estimate additional primary-school classrooms.

**Assumptions:** Population figures are in thousands. Kampala, Wakiso and Gulu use the supplied illustrative data. Mbarara and Mbale are added synthetic examples, not official UBOS estimates. The fixed random seed is 42.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statistics import mean, median, variance, stdev, pvariance, pstdev

# Make src/ importable when this notebook is run from the repository root or its folder.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.population_forecaster import (
    DistrictPopulation, Forecaster, LinearTrendForecaster,
    CAGRForecaster, FibonacciRatioForecaster,
    calculate_metrics, bootstrap_prediction_interval
)

SEED = 42
YEARS = np.arange(2015, 2025)
POPULATION_DATA = {
    "Kampala": [1200, 1250, 1300, 1350, 1420, 1500, 1580, 1650, 1720, 1800],
    "Wakiso":  [950, 1000, 1070, 1150, 1220, 1300, 1390, 1480, 1570, 1670],
    "Gulu":    [320, 330, 345, 360, 375, 390, 410, 430, 455, 480],
    # Additional illustrative series created for this assignment:
    "Mbarara": [195, 202, 210, 219, 228, 238, 249, 260, 273, 286],
    "Mbale":   [185, 190, 197, 204, 212, 220, 229, 238, 248, 259],
}
districts = {
    name: DistrictPopulation(name, YEARS.copy(), np.array(values, dtype=float))
    for name, values in POPULATION_DATA.items()
}
for district in districts.values():
    print(district)

## Task 1 — Class design and validation

`DistrictPopulation` stores years and population estimates as NumPy arrays. Its constructor validates equal lengths, one-dimensional inputs, finite/non-negative population values, unique years, and chronological order. `__repr__` gives a compact summary and `__len__` returns the number of observations.

In [ ]:
# Demonstrate validation (expected ValueError):
try:
    DistrictPopulation("Invalid", np.array([2020, 2021]), np.array([100, -1]))
except ValueError as exc:
    print("Validation works:", exc)

## Task 2 — Descriptive statistics: `statistics` versus NumPy

`statistics.variance` and `statistics.stdev` calculate **sample** variance and standard deviation, dividing by *n − 1*. NumPy's `np.var` and `np.std` default to population statistics, dividing by *n* (`ddof=0`). Setting `ddof=1` makes NumPy use the sample denominator *n − 1*. Means and medians should match, subject to normal floating-point rounding.

In [ ]:
stats_rows = []
for name, district in districts.items():
    x = district.populations.tolist()
    arr = district.populations
    stats_rows.append({
        "District": name,
        "statistics mean": mean(x),
        "NumPy mean": float(np.mean(arr)),
        "statistics median": median(x),
        "NumPy median": float(np.median(arr)),
        "statistics sample variance": variance(x),
        "NumPy variance ddof=1": float(np.var(arr, ddof=1)),
        "NumPy population variance ddof=0": float(np.var(arr, ddof=0)),
        "statistics sample SD": stdev(x),
        "NumPy SD ddof=1": float(np.std(arr, ddof=1)),
    })
stats_table = pd.DataFrame(stats_rows).set_index("District")
display(stats_table.round(3))

## Task 3 — Year-on-year growth and CAGR

Year-on-year growth is `(current / previous − 1) × 100`. CAGR summarises the change from the first to the last observation over nine elapsed years. These are relative growth measures, so the fastest-growing district need not have the largest absolute population increase.

In [ ]:
growth_rows = []
growth_by_district = {}
for name, district in districts.items():
    values = district.populations
    yoy = (values[1:] / values[:-1] - 1) * 100
    cagr = ((values[-1] / values[0]) ** (1 / (district.years[-1] - district.years[0])) - 1) * 100
    growth_by_district[name] = yoy
    growth_rows.append({
        "District": name,
        "Mean YoY growth (%)": float(np.mean(yoy)),
        "Minimum YoY (%)": float(np.min(yoy)),
        "Maximum YoY (%)": float(np.max(yoy)),
        "CAGR 2015–2024 (%)": float(cagr),
        "Absolute change (thousands)": float(values[-1] - values[0]),
    })
growth_table = pd.DataFrame(growth_rows).set_index("District").sort_values("CAGR 2015–2024 (%)", ascending=False)
display(growth_table.round(2))
print("Fastest in relative terms by CAGR:", growth_table.index[0])

## Task 4 — Forecasting models

All models implement the abstract `Forecaster` interface with `fit()` and `predict(horizon)`.

- **Linear trend:** least-squares straight line estimated with `np.polyfit`.
- **CAGR:** assumes a constant compound annual growth rate.
- **Fibonacci ratio:** a deliberately naive benchmark that multiplies each future value by successive ratios of Fibonacci numbers (3/2, 5/3, 8/5, …). These ratios converge to the golden ratio (~1.618), which is not a realistic annual population-growth assumption. The model is included for critical comparison, not as a recommended demographic model.

In [ ]:
MODEL_FACTORIES = {
    "Linear trend": LinearTrendForecaster,
    "CAGR": CAGRForecaster,
    "Fibonacci ratio": FibonacciRatioForecaster,
}

# Demonstrate use of fit/predict for each model on one district.
demo = districts["Kampala"]
for model_name, model_factory in MODEL_FACTORIES.items():
    fitted_model = model_factory().fit(demo.years, demo.populations)
    print(f"{model_name}: 2025–2027 predictions (thousands) = {np.round(fitted_model.predict(3), 2)}")

## Task 5 — Train/test validation (train 2015–2021; test 2022–2024)

Each model is fitted only on 2015–2021, then predicts three years. MAE, RMSE and MAPE compare predictions with the held-out actual values. The model with the **lowest test RMSE** is selected for each district; the full comparison table is printed for each district.

In [ ]:
TRAIN_END_YEAR = 2021
HOLDOUT_YEARS = np.array([2022, 2023, 2024])
HORIZON_TEST = len(HOLDOUT_YEARS)

validation_records = []
selected_model_names = {}
for name, district in districts.items():
    train_mask = district.years <= TRAIN_END_YEAR
    test_mask = district.years > TRAIN_END_YEAR
    train_years, train_values = district.years[train_mask], district.populations[train_mask]
    test_values = district.populations[test_mask]
    district_rows = []
    for model_name, model_factory in MODEL_FACTORIES.items():
        model = model_factory().fit(train_years, train_values)
        predictions = model.predict(HORIZON_TEST)
        metrics = calculate_metrics(test_values, predictions)
        row = {"District": name, "Model": model_name, **metrics}
        district_rows.append(row)
        validation_records.append(row)
    district_table = pd.DataFrame(district_rows).sort_values("RMSE")
    selected_model_names[name] = str(district_table.iloc[0]["Model"])
    print(f"\n{name}: 2022–2024 test results")
    display(district_table.set_index("Model").round(3))
    print("Selected model (lowest RMSE):", selected_model_names[name])

validation_table = pd.DataFrame(validation_records)
print("Overall validation comparison:")
display(validation_table.pivot(index="District", columns="Model", values="RMSE").round(2))

## Task 6 — Forecast 2025–2029 and compare variances

After model selection, each chosen model is refitted to all observations from 2015–2024. Forecast variance is compared with the variance of the observed 2015–2024 series. A lower forecast variance can indicate that a model projects a smoother or more stable path; a higher variance can indicate accelerating or more dispersed values. Variances here describe variation in the series, not uncertainty around the forecast.

In [ ]:
FORECAST_YEARS = np.arange(2025, 2030)
forecast_results = {}
variance_rows = []
for name, district in districts.items():
    model_name = selected_model_names[name]
    model = MODEL_FACTORIES[model_name]().fit(district.years, district.populations)
    point, lower, upper = bootstrap_prediction_interval(
        model, horizon=5, n_resamples=1000, seed=SEED
    )
    forecast_results[name] = {
        "model_name": model_name, "model": model,
        "point": point, "lower": lower, "upper": upper
    }
    variance_rows.append({
        "District": name,
        "Selected model": model_name,
        "Actual variance (2015–2024)": float(np.var(district.populations, ddof=0)),
        "Forecast variance (2025–2029)": float(np.var(point, ddof=0)),
        "Forecast minus actual variance": float(np.var(point, ddof=0) - np.var(district.populations, ddof=0)),
    })
    print(f"\n{name} forecast (population in thousands)")
    display(pd.DataFrame({
        "Year": FORECAST_YEARS, "Forecast": point,
        "95% lower": lower, "95% upper": upper
    }).round(2))
variance_table = pd.DataFrame(variance_rows).set_index("District")
display(variance_table.round(2))

## Extension — Bootstrap prediction intervals

The code resamples fitted-model residuals with replacement at least 1,000 times using `np.random.default_rng(seed=42)`. Each sampled residual sequence is cumulatively added to the point forecast to form a simulated path; the 2.5th and 97.5th percentiles give empirical 95% intervals. This simple method assumes historical residuals are informative about future errors. It does not account for structural changes, migration shocks, changing fertility/mortality, or uncertainty in the model choice.

## Task 7 — Plot actual, fitted and forecast values

Each district has a subplot. The vertical dashed line marks the training/test boundary after 2021; the shaded band is the residual-bootstrap 95% prediction interval for 2025–2029.

In [ ]:
fig, axes = plt.subplots(len(districts), 1, figsize=(11, 4 * len(districts)), sharex=True)
if len(districts) == 1:
    axes = [axes]

for ax, (name, district) in zip(axes, districts.items()):
    result = forecast_results[name]
    model = result["model"]
    # Reconstruct fitted values using the selected full-data model.
    fitted = model.fitted_
    ax.plot(district.years, district.populations, marker="o", label="Actual 2015–2024")
    ax.plot(district.years, fitted, linestyle="--", label=f"Fitted: {result['model_name']}")
    ax.plot(FORECAST_YEARS, result["point"], marker="o", label="Forecast 2025–2029")
    ax.fill_between(FORECAST_YEARS, result["lower"], result["upper"], alpha=0.2, label="95% bootstrap interval")
    ax.axvline(2021.5, linestyle=":", color="black", label="Train/test split")
    ax.set_title(name)
    ax.set_ylabel("Population (thousands)")
    ax.grid(True, alpha=0.25)
    ax.legend(loc="best", fontsize=8)

axes[-1].set_xlabel("Year")
plt.tight_layout()
plt.show()

## Task 8 — Primary-school classroom planning output

Assume 18% of the population is of primary-school age and each classroom accommodates 53 pupils. Because the population series is in thousands, forecast population is multiplied by 1,000 before estimating pupils. “Additional classrooms by 2029” is calculated as 2029 classroom demand minus 2024 demand, rounded up. Negative results are set to zero because this task asks for additional classrooms rather than surplus capacity. This is a simplified planning estimate: it assumes every school-age child requires a classroom place and does not account for enrolment, shifts, existing capacity, class-size policy, or distribution within districts.

In [ ]:
CLASSROOM_SHARE = 0.18
PUPILS_PER_CLASSROOM = 53

planning_rows = []
for name, district in districts.items():
    forecast_2029_thousands = float(forecast_results[name]["point"][-1])
    population_2024 = float(district.populations[-1]) * 1000
    population_2029 = forecast_2029_thousands * 1000
    pupils_2024 = population_2024 * CLASSROOM_SHARE
    pupils_2029 = population_2029 * CLASSROOM_SHARE
    classrooms_2024 = int(np.ceil(pupils_2024 / PUPILS_PER_CLASSROOM))
    classrooms_2029 = int(np.ceil(pupils_2029 / PUPILS_PER_CLASSROOM))
    additional = max(0, classrooms_2029 - classrooms_2024)
    planning_rows.append({
        "District": name,
        "Selected model": selected_model_names[name],
        "2024 population": population_2024,
        "2029 forecast population": population_2029,
        "2024 school-age pupils (18%)": pupils_2024,
        "2029 school-age pupils (18%)": pupils_2029,
        "Classrooms needed in 2024": classrooms_2024,
        "Classrooms needed in 2029": classrooms_2029,
        "Additional classrooms by 2029": additional,
    })
planning_table = pd.DataFrame(planning_rows).set_index("District")
display(planning_table.round(0).astype({
    "2024 population": int,
    "2029 forecast population": int,
    "2024 school-age pupils (18%)": int,
    "2029 school-age pupils (18%)": int,
    "Classrooms needed in 2024": int,
    "Classrooms needed in 2029": int,
    "Additional classrooms by 2029": int,
}))

## Findings & Limitations (150–300 words)

The descriptive statistics summarise both the level and dispersion of each district's illustrative population series. Mean and median are measures of central tendency, while variance and standard deviation quantify dispersion. `statistics.variance` and `statistics.stdev` use sample estimators with denominator *n − 1*; NumPy defaults to population estimators with denominator *n*, so `ddof=1` is needed to match the sample statistics. Relative growth is best compared using CAGR, while absolute increases remain important for allocating resources.

The holdout comparison tests whether each model can reproduce the final three observed years when trained only on 2015–2021. Selecting by RMSE penalises larger forecast errors more strongly than MAE, while MAPE expresses errors relative to observed values. The best model can differ by district, and a model winning on only three test years should not be treated as definitively superior. Linear trend assumes a constant absolute annual increase; CAGR assumes a constant proportional rate. Both can fail when urbanisation, migration, policy, or boundary definitions change.

The Fibonacci-ratio model is not a credible population forecast under ordinary demographic assumptions. Its annual multipliers converge to approximately 1.618, implying growth near 62% per year. It is defensible only as a mathematical benchmark or in an unusual system explicitly governed by those ratios, not as a realistic district population model. Bootstrap intervals reflect resampled historical residuals and may understate uncertainty when errors are autocorrelated or future conditions change. The classroom estimates are planning approximations, not official school-capacity requirements: the 18% share, class size, and synthetic extra-district data are assumptions. A real plan should use current UBOS projections, district-level age structures, enrolment rates, existing classrooms, and local school distribution.